In [37]:
import yfinance as yf
import pandas as pd
import talib
import torch
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [101]:
data = yf.download("^GDAXI", start="1990-01-01", end="2024-01-01")
data.to_csv('index_stock_1990.csv')

df = pd.read_csv('index_stock_1990.csv')
print(df.shape)
print(df.head())


[*********************100%***********************]  1 of 1 completed

(8602, 6)
        Price               Close                High                 Low  \
0      Ticker              ^GDAXI              ^GDAXI              ^GDAXI   
1        Date                 NaN                 NaN                 NaN   
2  1990-01-02  1788.8900146484375  1788.8900146484375  1788.8900146484375   
3  1990-01-03     1867.2900390625     1867.2900390625     1867.2900390625   
4  1990-01-04  1830.9200439453125  1830.9200439453125  1830.9200439453125   

                 Open  Volume  
0              ^GDAXI  ^GDAXI  
1                 NaN     NaN  
2  1788.8900146484375       0  
3     1867.2900390625       0  
4  1830.9200439453125       0  


In [137]:
df = df.iloc[2:]
df = df[df['Volume'] != 0]

In [138]:
df

,index,Date,Close,High,Low,Open,Volume,Pattern
2463,2465,1999-11-01,0.271867,0.269832,0.270648,0.268787,0.024949,1
2464,2466,1999-11-02,0.273290,0.269962,0.269181,0.269150,0.055540,0
2465,2467,1999-11-03,0.274189,0.272529,0.270977,0.269639,0.079168,0
2466,2468,1999-11-04,0.279019,0.276273,0.274825,0.271997,0.083509,0
2467,2469,1999-11-05,0.280471,0.278176,0.277175,0.276536,0.069704,0
...,...,...,...,...,...,...,...,...
8594,8596,2023-12-20,0.996034,0.986401,0.995755,0.989164,0.124200,0
8595,8597,2023-12-21,0.993086,0.981198,0.991196,0.981993,0.113430,0
8596,8598,2023-12-22,0.994298,0.982917,0.992985,0.982376,0.090740,0
8597,8599,2023-12-27,0.996617,0.985492,0.995950,0.985861,0.073852,0


In [139]:
df['Doji'] = talib.CDLDOJI(df['Open'], df['High'], df['Low'], df['Close'])
df['Hammer'] = talib.CDLHAMMER(df['Open'], df['High'], df['Low'], df['Close'])
df['Engulfing'] = talib.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])


In [140]:
def min_max_normalization(x, columns=[]):
    x = x.astype(float)
    x_scaled = (x - x.min()) / (x.max() - x.min())
    return x_scaled


In [141]:
df[['Close', 'High', 'Low', 'Open', 'Volume']] = df[['Close', 'High', 'Low', 'Open', 'Volume']].apply(min_max_normalization)

In [142]:
df = df.rename(columns = {'Price':'Date'})

In [143]:
df = df.reset_index()

In [144]:
df.drop(columns='index')
df.head(20)

,level_0,index,Date,Close,High,Low,Open,Volume,Pattern,Doji,Hammer,Engulfing
0,2463,2465,1999-11-01,0.227665,0.219982,0.227022,0.224809,0.023582,1,0,0,0
1,2464,2466,1999-11-02,0.229174,0.220121,0.225467,0.225194,0.054216,0,0,0,0
2,2465,2467,1999-11-03,0.230128,0.222863,0.227371,0.225713,0.077877,0,0,0,0
3,2466,2468,1999-11-04,0.235251,0.226863,0.231449,0.228212,0.082224,0,0,0,0
4,2467,2469,1999-11-05,0.236792,0.228896,0.233940,0.233025,0.068400,0,0,0,0
5,2468,2470,1999-11-08,0.236095,0.228920,0.235064,0.232269,0.044334,0,0,0,0
6,2469,2471,1999-11-09,0.239302,0.233975,0.237835,0.234005,0.063027,0,0,0,0
7,2470,2472,1999-11-10,0.242570,0.233101,0.238339,0.236119,0.062488,0,0,0,0
8,2471,2473,1999-11-11,0.246678,0.237284,0.241963,0.238084,0.070902,0,0,0,0
9,2472,2474,1999-11-12,0.245903,0.238677,0.244445,0.243172,0.067727,1,0,0,0


In [145]:
df.value_counts(['Doji', 'Hammer', 'Engulfing'])

Doji  Hammer  Engulfing
0     0        0           5107
               100          243
      100      0            209
100   0        0             88
      100      0             15
0     0       -100            7
      100      100            1
Name: count, dtype: int64

In [146]:
def define_pattern(x):
    # No pattern
    if x['Doji'] == 0 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 0
    
    # Doji
    elif x['Doji'] == 100 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 1
    
    # Hammer
    elif x['Doji'] == 0 and x['Hammer'] == 100 and x['Engulfing'] == 0:
        return 2
    
    # Engulfing
    elif x['Doji'] == 0 and x['Hammer'] == 0 and (x['Engulfing'] != 0):
        return 3
    
    # 2 classes at once
    else:
        return -1 
    


In [147]:
df['Pattern'] = df.apply(define_pattern, axis=1)

In [148]:
df

,level_0,index,Date,Close,High,Low,Open,Volume,Pattern,Doji,Hammer,Engulfing
0,2463,2465,1999-11-01,0.227665,0.219982,0.227022,0.224809,0.023582,0,0,0,0
1,2464,2466,1999-11-02,0.229174,0.220121,0.225467,0.225194,0.054216,0,0,0,0
2,2465,2467,1999-11-03,0.230128,0.222863,0.227371,0.225713,0.077877,0,0,0,0
3,2466,2468,1999-11-04,0.235251,0.226863,0.231449,0.228212,0.082224,0,0,0,0
4,2467,2469,1999-11-05,0.236792,0.228896,0.233940,0.233025,0.068400,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5665,8594,8596,2023-12-20,0.995794,0.985472,0.995501,0.988513,0.122972,0,0,0,0
5666,8595,8597,2023-12-21,0.992666,0.979914,0.990669,0.980910,0.112187,0,0,0,0
5667,8596,8598,2023-12-22,0.993952,0.981751,0.992565,0.981316,0.089466,0,0,0,0
5668,8597,8599,2023-12-27,0.996412,0.984502,0.995708,0.985010,0.072553,0,0,0,0


In [149]:
df['Pattern'].value_counts()

Pattern
 0    5107
 3     250
 2     209
 1      88
-1      16
Name: count, dtype: int64

In [153]:
df = df[df['Pattern'] != -1]
# df = df.drop(columns =['Doji', 'Hammer', 'Engulfing'])

In [154]:
df.head(20)

,level_0,index,Date,Close,High,Low,Open,Volume,Pattern
0,2463,2465,1999-11-01,0.227665,0.219982,0.227022,0.224809,0.023582,0
1,2464,2466,1999-11-02,0.229174,0.220121,0.225467,0.225194,0.054216,0
2,2465,2467,1999-11-03,0.230128,0.222863,0.227371,0.225713,0.077877,0
3,2466,2468,1999-11-04,0.235251,0.226863,0.231449,0.228212,0.082224,0
4,2467,2469,1999-11-05,0.236792,0.228896,0.233940,0.233025,0.068400,0
5,2468,2470,1999-11-08,0.236095,0.228920,0.235064,0.232269,0.044334,0
6,2469,2471,1999-11-09,0.239302,0.233975,0.237835,0.234005,0.063027,0
7,2470,2472,1999-11-10,0.242570,0.233101,0.238339,0.236119,0.062488,0
8,2471,2473,1999-11-11,0.246678,0.237284,0.241963,0.238084,0.070902,0
9,2472,2474,1999-11-12,0.245903,0.238677,0.244445,0.243172,0.067727,0


## Building neural network with Torch

In [155]:
W1 = torch.randn(5,8,requires_grad = True)
b1 = torch.zeros(8,requires_grad = True)
W2 = torch.randn(8,4,requires_grad = True)
b2 = torch.zeros(4,requires_grad = True)

In [156]:
def forward_pass(X):
    hidden_lay_pre_act = torch.matmul(X, W1) + b1
    relu_act = torch.relu(hidden_lay_pre_act)
    logits = torch.matmul(relu_act, W2) + b2
    return logits

In [157]:
freq = df['Pattern'].value_counts().sort_index()
weights = 1/(freq**0.5)

In [158]:
tensor_1 = torch.tensor(weights.values, dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight = tensor_1)

In [159]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)

In [160]:
for epoch in range(100):
    logits = forward_pass(torch.tensor(df[['Close', 'High', 'Low', 'Open', 'Volume']].values, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(df['Pattern'].values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

3.7918031215667725
1.6558161973953247
1.043971300125122
1.0834851264953613
1.049658179283142
1.0121407508850098
1.010400652885437
1.0063403844833374
1.004002571105957
1.002276062965393


## Doing using normal train-test split

In [129]:
# df = df[df['Pattern'] != 3]

In [161]:
df_train, df_test = train_test_split(df, test_size = 0.2)

In [162]:
X_train = df_train[['Close', 'High', 'Low', 'Open', 'Volume']]
X_test = df_test[['Close', 'High', 'Low', 'Open', 'Volume']]
y_train = df_train['Pattern']
y_test = df_test['Pattern']


In [163]:
scaler_normal = StandardScaler()
X_train_scaled = scaler_normal.fit_transform(X_train)
X_test_scaled = scaler_normal.transform(X_test)

In [164]:
for epoch in range(500):
    logits = forward_pass(torch.tensor(X_train_scaled, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(y_train.values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [165]:
logits_test = forward_pass(torch.tensor(X_test_scaled, dtype=torch.float32))
logits_test

tensor([[ 2.1558,  0.4809,  0.7327,  1.0337],
        [ 3.1520,  1.5510,  1.6277,  1.7819],
        [ 3.0964, -0.4100,  0.6274,  0.9770],
        ...,
        [ 0.3540, -1.1955, -0.5992, -0.8365],
        [ 1.9514,  0.3984,  0.4100,  0.8205],
        [ 2.1296,  0.5843,  0.6432,  1.0196]], grad_fn=<AddBackward0>)

In [166]:
test_result = torch.argmax(logits_test, dim=1)
test_result

tensor([0, 0, 0,  ..., 0, 0, 0])

In [167]:
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94      1011
           1       0.00      0.00      0.00        17
           2       0.00      0.00      0.00        49
           3       0.00      0.00      0.00        54

    accuracy                           0.89      1131
   macro avg       0.22      0.25      0.24      1131
weighted avg       0.80      0.89      0.84      1131



C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [30]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [31]:
for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

1.3683979511260986
1.2354280948638916
1.3413268327713013
1.187791347503662
1.4376529455184937
1.2756632566452026
1.5361380577087402
1.3462456464767456
1.5045526027679443
1.3405340909957886
1.4318156242370605
1.5218842029571533
1.264201283454895
1.6812466382980347
1.4126776456832886
1.3182190656661987
1.2217377424240112
1.598706603050232
1.1765296459197998
1.2182198762893677
1.2267177104949951
1.1418014764785767
1.1102255582809448
1.308485746383667
1.1208410263061523
1.1444456577301025
2.3586888313293457
1.9335418939590454
1.1118903160095215
1.2206823825836182
1.2584141492843628
1.2579829692840576
1.4994057416915894
1.2152514457702637
1.3778671026229858
1.512357473373413
1.0858149528503418
1.2391334772109985
1.187421202659607
1.1717884540557861
1.2209758758544922
1.417731523513794
1.435203194618225
1.124928593635559
1.1459952592849731
1.2412678003311157
1.3177968263626099
1.5665514469146729
1.4605919122695923
1.281632661819458
1.3232190608978271
1.2374179363250732
1.512174367904663
1.31

In [32]:
logits_test_batch = forward_pass(torch.tensor(X_test.values, dtype=torch.float32))
test_result = torch.argmax(logits_test_batch, dim=1)
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.77      0.27      0.40       195
           1       0.14      0.61      0.23        31
           2       0.00      0.00      0.00        10
           3       0.07      0.17      0.10        18

    accuracy                           0.30       254
   macro avg       0.24      0.26      0.18       254
weighted avg       0.61      0.30      0.34       254



In [69]:
y_train.value_counts()

Pattern
0    802
1    146
3     55
2     12
Name: count, dtype: int64

In [ ]:
sm = SMOTE(sampling_strategy={1:400, 2:400, 3:400})
X_train_partial_sm, y_train_partial_sm = 

## Applying imbalanced learn

In [33]:
sm = SMOTE()
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

In [36]:
y_train_sm.value_counts()

Pattern
3    802
0    802
1    802
2    802
Name: count, dtype: int64

In [51]:
scaler = StandardScaler()
scaler_1 = scaler.fit(X_train)
X_train_sm = scaler_1.transform(X_train_sm)
X_test_sm = scaler_1.transform(X_test)

C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [52]:
X_train_sm

array([[-7.75081514, -7.91561079, -7.72754202, -7.95364686, -2.7357407 ],
       [-3.02871979, -2.86096908, -3.01902505, -2.73275032, 23.47615795],
       [-7.3642248 , -7.57359788, -7.33458034, -7.54779337, -5.01459387],
       ...,
       [-6.42249139, -6.29779174, -6.43242183, -6.15723151, -1.52892429],
       [ 1.04930922,  1.16305193,  1.09800945,  1.24767119, -8.80415984],
       [ 2.82987141,  2.79674964,  2.63657154,  2.73010178, -3.59885657]],
      shape=(3208, 5))

In [53]:
X_train_sm = torch.tensor(X_train_sm, dtype=torch.float32)
y_train_sm = torch.tensor(y_train_sm, dtype=torch.long)

dataset = TensorDataset(X_train_sm, y_train_sm)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

C:\Users\USER\AppData\Local\Temp\ipykernel_40268\2942018516.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_sm = torch.tensor(y_train_sm, dtype=torch.long)


In [54]:
 for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

100.02243041992188
90.01861572265625
87.29467010498047
20.76238441467285
9.085517883300781
55.10868453979492
65.02256774902344
94.97692108154297
26.068044662475586
18.887218475341797
22.610946655273438
14.75804328918457
26.334909439086914
41.625282287597656
11.44057846069336
6.331490516662598
10.396130561828613
26.401823043823242
32.797176361083984
4.529830455780029
20.17009735107422
12.093696594238281
6.470258712768555
11.074352264404297
17.841598510742188
13.091679573059082
12.782931327819824
7.6161298751831055
4.1518449783325195
23.18633270263672
9.911158561706543
19.60047149658203
1.5992647409439087
5.369505405426025
4.7104902267456055
6.402195453643799
6.185781955718994
2.139009714126587
2.840399980545044
6.142848014831543
4.836019039154053
10.545116424560547
5.3059892654418945
9.269389152526855
3.7641329765319824
1.4361767768859863
2.164360284805298
0.9086177945137024
4.345366477966309
4.354528903961182
5.329564094543457
2.763134717941284
5.77982759475708
6.448581695556641
2.6284

In [58]:
logits_test_batch_sm = forward_pass(torch.tensor(X_test_sm,dtype=torch.float32))
test_result_sm = torch.argmax(logits_test_batch_sm, dim=1)
print(classification_report(y_test, test_result_sm.numpy()))

              precision    recall  f1-score   support

           0       1.00      0.01      0.02       195
           1       0.30      0.77      0.43        31
           2       0.21      0.30      0.25        10
           3       0.10      0.89      0.18        18

    accuracy                           0.18       254
   macro avg       0.40      0.49      0.22       254
weighted avg       0.82      0.18      0.09       254



## Using partial smote for class 1,2,3

In [72]:
sm = SMOTE(sampling_strategy={1:400, 2:400, 3:400})
X_train_partial_sm, y_train_partial_sm= sm.fit_resample(X_train_scaled, y_train)

In [77]:
X_train_partial_sm_tensor = torch.tensor(X_train_partial_sm, dtype=torch.float32)
y_train_partial_sm_tensor = torch.tensor(y_train_partial_sm, dtype=torch.long)

dataset_partial_sm = TensorDataset(X_train_partial_sm_tensor, y_train_partial_sm_tensor)
loader_partial_sm = DataLoader(dataset_partial_sm, batch_size=32, shuffle=True)

In [78]:
 for epoch in range(500):
    for X_batch, y_batch in loader_partial_sm:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

0.5365168452262878
0.5667194724082947
0.372213751077652
0.398631751537323
0.3448191285133362
0.32930341362953186
0.37598511576652527
0.3089423179626465
0.5038843154907227
0.514828622341156
0.3306044638156891
0.3332614600658417
0.30531612038612366
0.29381510615348816
0.3962377607822418
0.28689271211624146
0.5282073616981506
0.26010456681251526
0.18885383009910583
0.2573309540748596
0.19917571544647217
0.20778335630893707
0.3206399381160736
0.4233512580394745
0.22318045794963837
0.47875508666038513
0.29191264510154724
0.38428670167922974
0.3808315396308899
0.26172342896461487
0.2684502601623535
0.33084097504615784
0.40421977639198303
0.26417815685272217
0.2827073335647583
0.3649449646472931
0.27155542373657227
0.4881562292575836
0.24293842911720276
0.3636889159679413
0.4397585093975067
0.2531244456768036
0.1994735449552536
0.45326802134513855
0.44378724694252014
0.3358626663684845
0.32757478952407837
0.40212705731391907
0.3083493709564209
0.6181771159172058
0.4320354759693146
0.240326151

In [80]:
logits_test_batch_partial_sm = forward_pass(torch.tensor(X_test_scaled,dtype=torch.float32))
test_result_partial_sm = torch.argmax(logits_test_batch_partial_sm, dim=1)
print(classification_report(y_test, test_result_partial_sm.numpy()))

              precision    recall  f1-score   support

           0       0.84      0.16      0.27       195
           1       0.24      0.42      0.31        31
           2       0.25      0.10      0.14        10
           3       0.08      0.72      0.15        18

    accuracy                           0.23       254
   macro avg       0.35      0.35      0.22       254
weighted avg       0.69      0.23      0.26       254



There are too little data for 2 and 3 in reality, so SMOTE for new data is not possible. Tried and harmed the model. 
- Deletion is also not possible, because it would turn into binary classification which kind of useless